In [20]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

BASE = "../data/raw/situnes/SiTunes"
OUT  = "../data/processed"
Path(OUT).mkdir(parents=True, exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────
df1    = pd.read_csv(f"{BASE}/Stage1/interactions.csv")
df2    = pd.read_csv(f"{BASE}/Stage2/interactions.csv")
df3    = pd.read_csv(f"{BASE}/Stage3/interactions.csv")
wrist2 = np.load(f"{BASE}/Stage2/wrist.npy", allow_pickle=True)
wrist3 = np.load(f"{BASE}/Stage3/wrist.npy", allow_pickle=True)

with open(f"{BASE}/Stage2/env.json") as f: env2 = json.load(f)
with open(f"{BASE}/Stage3/env.json") as f: env3 = json.load(f)

# Recover 9 missing track IDs from music_info.csv
music_withname = pd.read_csv(f"{BASE}/music_metadata/music_info_withname.csv")
music_base     = pd.read_csv(f"{BASE}/music_metadata/music_info.csv")
_missing_ids   = [3, 42, 293, 401, 421, 453, 513, 740, 928]
_missing_rows  = music_base[music_base["item_id"].isin(_missing_ids)].copy()
_missing_rows["i_id_c"]           = _missing_rows["item_id"].astype(float)
_missing_rows["music"]            = "Unknown"
_missing_rows["singer"]           = "Unknown"
_missing_rows["general_genre"]    = "other"
_missing_rows["general_genre_id"] = 0
_missing_rows  = _missing_rows.drop(columns=["item_id"])
music = pd.concat([music_withname, _missing_rows], ignore_index=True)

# ── Drop unused columns ───────────────────────────────────────────────────
df2 = df2.drop(columns=["duration", "rec_type"])
df3 = df3.drop(columns=["duration", "rec_type"])
df1 = df1.drop(columns=["duration"])

# ── Drop interactions with missing music (should be 0) ───────────────────
valid_ids = set(music["i_id_c"].dropna().astype(int))
df2 = df2[df2.item_id.isin(valid_ids)].reset_index(drop=True)
df3 = df3[df3.item_id.isin(valid_ids)].reset_index(drop=True)
print(f"df2: {len(df2)} rows, df3: {len(df3)} rows")  # expect 897, 509

# ── Clip emotion values ───────────────────────────────────────────────────
for col in ["emo_pre_valence", "emo_pre_arousal",
            "emo_post_valence", "emo_post_arousal"]:
    df2[col] = df2[col].clip(-0.99, 0.99)
    df3[col] = df3[col].clip(-0.99, 0.99)

# ── Reward signal ─────────────────────────────────────────────────────────
def compute_reward(row, threshold=0.1):
    score = (row.emo_post_valence - row.emo_pre_valence) * 0.7 + \
            (row.emo_post_arousal - row.emo_pre_arousal) * 0.3
    return 1 if score > threshold else (-1 if score < -threshold else 0)

df2["reward"] = df2.apply(compute_reward, axis=1)
df3["reward"] = df3.apply(compute_reward, axis=1)
print("df2 rewards:", df2.reward.value_counts().sort_index().to_dict())
print("df3 rewards:", df3.reward.value_counts().sort_index().to_dict())

# ── Music action buckets ──────────────────────────────────────────────────
music = music.dropna(subset=["valence"])

def get_action_bucket(valence, energy, tempo):
    v = 0 if valence < 0.33 else 1   # 0=low, 1=medium/high
    e = 0 if energy  < 0.4  else 1   # 0=low, 1=high
    t = 0 if tempo   < 100  else 1   # 0=slow, 1=fast
    return int(v * 4 + e * 2 + t)    # always 0-7

music["action_bucket"] = music.apply(
    lambda r: get_action_bucket(r["valence"], r["energy"], r["tempo"]), axis=1
)
print("Music buckets:", music.action_bucket.value_counts().sort_index().to_dict())

# ── Add action_bucket and obs_idx to interactions ─────────────────────────
bucket_map = dict(zip(music["i_id_c"].astype(int), music["action_bucket"]))
df2["action_bucket"] = df2["item_id"].map(bucket_map).astype(int)
df3["action_bucket"] = df3["item_id"].map(bucket_map).astype(int)
df2["obs_idx"] = df2.index
df3["obs_idx"] = df3.index

# ── Encode wrist observations (0-179) ────────────────────────────────────
ACTIVITY_REMAP = {0: 0, 1: 1, 2: 2, 3: 0, 4: 3, 5: 4}

def encode_timestep(wrist_ts, time_bucket, weather_bucket):
    intensity = wrist_ts[1]
    activity  = ACTIVITY_REMAP.get(int(wrist_ts[3]), 0)
    ib = 0 if intensity < 10 else (1 if intensity < 30 else (2 if intensity < 80 else 3))
    wrist_obs = ib * 5 + activity                              # 0-19
    return int(wrist_obs * 9 + time_bucket * 3 + weather_bucket)  # 0-179

def encode_session(wrist_session, env_entry):
    tb = max(0, min(2, int(env_entry["time"]) - 1))
    wb = max(0, min(2, int(env_entry["weather"][0])))
    return [encode_timestep(wrist_session[t], tb, wb) for t in range(30)]

def encode_all(df, wrist, env):
    encoded = []
    for i, row in df.iterrows():
        key   = str(int(row["inter_id"]))
        entry = env[key] if key in env else {"time": 2, "weather": [0, 0, 20, 60]}
        encoded.append(encode_session(wrist[i], entry))
    return np.array(encoded, dtype=np.int32)

encoded2 = encode_all(df2, wrist2, env2)
encoded3 = encode_all(df3, wrist3, env3)
print(f"wrist2_encoded: {encoded2.shape}  range {encoded2.min()}-{encoded2.max()}")
print(f"wrist3_encoded: {encoded3.shape}  range {encoded3.min()}-{encoded3.max()}")

# ── Save ──────────────────────────────────────────────────────────────────
df2.to_csv(f"{OUT}/stage2_clean.csv",          index=False)
df3.to_csv(f"{OUT}/stage3_clean.csv",          index=False)
df1.to_csv(f"{OUT}/stage1_clean.csv",          index=False)
music.to_csv(f"{OUT}/music_situnes_clean.csv", index=False)
np.save(f"{OUT}/wrist2_encoded.npy", encoded2)
np.save(f"{OUT}/wrist3_encoded.npy", encoded3)
with open(f"{OUT}/env2_clean.json", "w") as f: json.dump(env2, f)
with open(f"{OUT}/env3_clean.json", "w") as f: json.dump(env3, f)

print("\n✓ Done. All files saved to", OUT)

df2: 897 rows, df3: 509 rows
df2 rewards: {-1: 167, 0: 425, 1: 305}
df3 rewards: {-1: 116, 0: 232, 1: 161}
Music buckets: {0: 35, 1: 83, 2: 27, 3: 157, 4: 17, 5: 38, 6: 134, 7: 453}
wrist2_encoded: (897, 30)  range 0-174
wrist3_encoded: (509, 30)  range 0-178

✓ Done. All files saved to ../data/processed
